In [ ]:
# Drive 마운트 & 의존성
from google.colab import drive
drive.mount('/content/drive')

# 과제 허용 범위: PyTorch/TorchVision, OpenCV, Pillow, pycocotools, WandB
!pip install -q wandb pyyaml pycocotools opencv-python pillow onnx onnxscript
!wandb login


In [ ]:
# 데이터셋 준비 (한 번만, 이미 있으면 스킵)
import os

DRIVE_DATA = "/content/drive/MyDrive/Data"

# --- VOC 2007 + 2012 ---
if not os.path.exists("/content/data/VOCdevkit/VOC2012"):
    !mkdir -p /content/data
    !cp {DRIVE_DATA}/VOCtrainval_06-Nov-2007.tar /content/data/
    !cp {DRIVE_DATA}/VOCtrainval_11-May-2012.tar /content/data/
    !tar -xf /content/data/VOCtrainval_06-Nov-2007.tar -C /content/data/
    !tar -xf /content/data/VOCtrainval_11-May-2012.tar -C /content/data/
    print("VOC 압축 해제 완료")
else:
    print("VOC 이미 있음 — 스킵")

# --- COCO 2017 images ---
if not os.path.exists("/content/coco/train2017"):
    !mkdir -p /content/coco
    !cp {DRIVE_DATA}/train2017.zip /content/coco/
    !unzip -q /content/coco/train2017.zip -d /content/coco/
    print("COCO images 압축 해제 완료")
else:
    print("COCO images 이미 있음 — 스킵")

# --- COCO annotations ---
if not os.path.exists("/content/coco/annotations"):
    !cp {DRIVE_DATA}/annotations_trainval2017.zip /content/coco/
    !unzip -q /content/coco/annotations_trainval2017.zip -d /content/coco/
    print("COCO annotations 완료")
else:
    print("COCO annotations 이미 있음 — 스킵")

# --- VOC-index pre-rendered masks ---
if not os.path.exists("/content/coco/voc_masks"):
    !tar -xf {DRIVE_DATA}/voc_masks.tar -C /content/coco/
    print("voc_masks 압축 해제 완료")
else:
    print("voc_masks 이미 있음 — 스킵")


In [ ]:
# 프로젝트 위치 확인 및 문법 체크
PROJECT_DIR = "/content/drive/MyDrive/Project01"
%cd $PROJECT_DIR

!python -m py_compile \
    src/train.py src/eval.py src/predict.py \
    src/models/deeplabv3plus.py src/models/export_onnx_structure.py \
    src/data/pascal_voc.py src/data/coco_voc.py src/data/transforms.py \
    src/utils/*.py src/tools/measure_gflops.py

!python src/tools/measure_gflops.py


In [ ]:
# 학습
%cd /content/drive/MyDrive/Project01

!python src/train.py --config src/semantic_segmentation.yaml


In [ ]:
# Evaluation
%cd /content/drive/MyDrive/Project01

!python src/eval.py \
    --config src/semantic_segmentation.yaml \
    --ckpt checkpoints/last_MobileNetV3_Large_ASPP256.pth

!python src/eval.py \
    --config src/semantic_segmentation.yaml \
    --ckpt checkpoints/best_ema_MobileNetV3_Large_ASPP256.pth \
    --use_ema


In [ ]:
# Inference: EMA checkpoint, single forward only (no TTA)
%cd /content/drive/MyDrive/Project01

!rm -rf /content/project01_submit_img /content/project01_pred
!mkdir -p /content/project01_submit_img /content/project01_pred submit/pred

# 테스트 이미지가 Drive/Data/img에 있다고 가정. 필요하면 경로만 수정하세요.
!cp -r /content/drive/MyDrive/Data/img/. /content/project01_submit_img/

!python src/predict.py \
    --ckpt checkpoints/best_ema_MobileNetV3_Large_ASPP256.pth \
    --img_dir /content/project01_submit_img \
    --pred_dir /content/project01_pred \
    --use_ema

!rm -rf submit/pred
!mkdir -p submit/pred
!cp -r /content/project01_pred/. submit/pred/
!ls submit/pred | head


In [ ]:
# ONNX 구조 export for GFLOPs site
%cd /content/drive/MyDrive/Project01

!python src/models/export_onnx_structure.py --output model_structure.onnx

import onnx, os
model = onnx.load("model_structure.onnx")
dims = [d.dim_value for d in model.graph.input[0].type.tensor_type.shape.dim]
print("input shape:", dims)
print("size bytes:", os.path.getsize("model_structure.onnx"))
